# Hunt 1: stack counting and the long tail

Hypothesis: attacker tooling is rare. Count every process across the estate and read the rarest first. Data is synthetic (seeded benign baseline plus the lab attack chain), so the result shows the method, not a real estate's noise.

In [ ]:
# Setup: run from the repository root or the notebooks/ folder. Lab mode is required
# because the cells generate synthetic attack telemetry (nothing is executed on a host).
import os
import sys
ROOT = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT)
os.environ.setdefault("CYBERPULSE_LAB_MODE", "true")

from datetime import datetime, timezone
from simulator.attack_chain import run_chain
from simulator.benign_baseline import generate_baseline
from soar import views  # noqa: F401  (used by the hunt cells)
from soar.soar_engine import SOAROrchestrator

orchestrator = SOAROrchestrator(policy_mode="DRY_RUN")
for event in generate_baseline(600, 1337):          # seeded benign traffic
    orchestrator.process_event(event)
run_chain(orchestrator, start=datetime(2026, 11, 1, 9, 0, tzinfo=timezone.utc))  # ten-stage synthetic intrusion
print(len(orchestrator.event_store.snapshot()), "events in the hunting store")

In [ ]:
result = views.stack(orchestrator, "process")
print(f"{result['distinct']} distinct processes, {result['tail']} in the long tail")
for row in result["rows"][:15]:
    print(f"{row['count']:>5}  {'TAIL' if row['tail'] else '    '}  alerted={row['alerted']}  {row['value']}")

Questions to ask of each tail value: which hosts and users ran it (pivot), is it signed, does its path make sense, did any rule fire on it?

In [ ]:
tail = [r["value"] for r in result["rows"] if r["tail"]]
for value in tail[:5]:
    p = views.pivot(orchestrator, "process", value)
    hosts = [h["value"] for h in p["related"].get("host", [])]
    print(value, "->", hosts)